# RoPE（旋转位置编码）教程

[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/transformer-tutorial/blob/main/rope_tutorial.ipynb)

本教程在 [KV Cache 教程](kv_cache_tutorial.ipynb) 的基础上，深入介绍 **RoPE**（Rotary Position Embedding，旋转位置编码）。

RoPE 是 LLaMA、GPT-NeoX、PaLM、Qwen、Mistral 等主流大语言模型采用的位置编码方案。  
其核心思想：用**旋转矩阵**将位置信息直接编码进 Query 和 Key，使得注意力分数天然具有**相对位置感知**能力，且与 KV Cache 无缝兼容。

## 目录
1. 位置编码方法对比（背景）
2. RoPE 数学原理
3. RoPE 实现：`precompute_rope_freqs` + `apply_rope`
4. 相对位置特性实验验证
5. RoPE 与 KV Cache 的天然兼容性
6. `CausalSelfAttentionRoPE` 实现
7. `GPTDecoderRoPE` 完整模型
8. 正确性验证（无缓存 == 有缓存）
9. 可视化：旋转角度、注意力分数衰减
10. 长度外推能力对比
11. 总结

## 环境与依赖

In [ ]:
import math
import time
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from typing import Optional, Tuple, List

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {DEVICE}')
print(f'PyTorch 版本: {torch.__version__}')


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 1. 位置编码方法对比

Transformer 本身没有顺序感，需要额外的位置编码来区分序列中不同位置的 token。

| 方法 | 代表模型 | 核心思路 | 优点 | 局限 |
|------|---------|---------|------|------|
| 正弦绝对 PE | 原始 Transformer | 固定 sin/cos 编码加到输入 | 无需训练 | 不感知相对距离 |
| 可学习绝对 PE | GPT-2, BERT | 每个位置一个可学习 embedding | 灵活 | 超出训练长度失效 |
| 相对 PE | Transformer-XL | 在注意力中加相对距离偏置 | 感知相对距离 | 实现复杂 |
| **RoPE** | **LLaMA, GPT-NeoX** | **旋转 Q/K 向量编码位置** | **天然相对感知 + KV Cache 友好** | 基础版外推有限 |
| ALiBi | MPT, BLOOM | 在分数上加线性距离惩罚 | 强外推能力 | 不感知具体位置值 |

### 绝对 PE 的核心问题

**问题 1**：绝对位置编码告诉模型『我在位置 5』，但语言理解更需要的是『我在前一个词后面 1 步』——**相对信息比绝对信息更重要**。

**问题 2**：可学习 PE 在训练时只见过位置 0 到 `max_len-1`，推理时遇到更长序列会崩溃。

In [ ]:
# ── 绝对 PE 的相对距离感知能力有限 ──────────────────────────────
# 用余弦相似度度量位置 i 和 j 的位置编码有多"相似"
# 理想情况：sim(PE(i), PE(j)) 只依赖 |i-j|，与绝对位置无关

def sinusoidal_pe(max_len, d_model):
    """原始 Transformer 的正弦位置编码"""
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(0, max_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe   # (max_len, d_model)


L, D = 32, 64
pe = sinusoidal_pe(L, D)   # (L, D)
pe_norm = F.normalize(pe, dim=-1)
sim_matrix = pe_norm @ pe_norm.T   # (L, L) 余弦相似度

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左图：相似度矩阵
ax = axes[0]
im = ax.imshow(sim_matrix.numpy(), cmap='RdBu', vmin=-0.5, vmax=1.0, aspect='auto')
plt.colorbar(im, ax=ax)
ax.set_xlabel('Position j'); ax.set_ylabel('Position i')
ax.set_title('Sinusoidal PE: Cosine Similarity Matrix\nsin(PE(i), PE(j))', fontsize=11)

# 右图：固定锚点，sim(PE(anchor), PE(j)) vs j
ax2 = axes[1]
for anchor in [0, 8, 16, 24]:
    ax2.plot(sim_matrix[anchor].numpy(), label=f'anchor={anchor}', linewidth=1.5)
ax2.set_xlabel('Position j'); ax2.set_ylabel('Cosine Similarity')
ax2.set_title('Similarity to Anchor vs Position\n(曲线形状因锚点不同而不同→绝对感知)', fontsize=11)
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.suptitle('Absolute Positional Encoding: Lacks Clean Relative Structure',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('abs_pe_problem.png', dpi=120, bbox_inches='tight')
plt.show()

print('观察：不同锚点的相似度曲线形状不同 → 不是纯粹的相对位置感知')

## 2. RoPE 数学原理

### 核心思想

RoPE 将位置信息通过**旋转**而非加法编码进 Query 和 Key：

$$\text{RoPE}(\mathbf{x}, m) = R_m \mathbf{x}$$

其中 $R_m$ 是一个与位置 $m$ 相关的旋转矩阵。

### 旋转矩阵构造

对于 $d_k$ 维向量，将其分为 $d_k/2$ 个维度对 $(x_{2i}, x_{2i+1})$，对每对应用 2D 旋转：

$$\begin{bmatrix} x_{2i}' \\ x_{2i+1}' \end{bmatrix} = \begin{bmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{bmatrix} \begin{bmatrix} x_{2i} \\ x_{2i+1} \end{bmatrix}$$

频率按几何级数递减（与原始 Transformer PE 一致）：

$$\theta_i = \frac{1}{10000^{2i/d_k}}, \quad i = 0, 1, \ldots, \frac{d_k}{2}-1$$

### 相对位置特性（关键定理）

$$\langle \text{RoPE}(\mathbf{q}, m),\; \text{RoPE}(\mathbf{k}, n) \rangle = f(\mathbf{q}, \mathbf{k},\; m - n)$$

**证明直觉（d_k=2 的情形）**：

$$\langle R_m\mathbf{q},\; R_n\mathbf{k} \rangle = \mathbf{q}^\top R_m^\top R_n \mathbf{k} = \mathbf{q}^\top R_{m-n} \mathbf{k}$$

旋转矩阵满足 $R_m^\top R_n = R_{n-m}$，所以内积只依赖于相对位置 $(m-n)$，与绝对位置无关。

### LLaMA 半分割约定

实践中（LLaMA 风格）常用更高效的**半分割**实现：

$$\text{RoPE}(\mathbf{x}, m) = \begin{bmatrix} \mathbf{x}_1 \odot \cos(m\boldsymbol{\theta}) - \mathbf{x}_2 \odot \sin(m\boldsymbol{\theta}) \\ \mathbf{x}_1 \odot \sin(m\boldsymbol{\theta}) + \mathbf{x}_2 \odot \cos(m\boldsymbol{\theta}) \end{bmatrix}$$

其中 $\mathbf{x}_1 = \mathbf{x}[:d/2]$，$\mathbf{x}_2 = \mathbf{x}[d/2:]$（与逐对旋转等价，只是维度排列不同）。

In [ ]:
# ══ RoPE 核心实现 ══════════════════════════════════════════════

def precompute_rope_freqs(
    d_k: int, max_len: int, theta: float = 10000.0
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    预计算 RoPE 的余弦/正弦频率表。

    频率: θᵢ = 1 / (theta ^ (2i / d_k)),  i = 0..d_k//2-1
    角度: angle[m, i] = m * θᵢ

    Returns:
        cos_table: (max_len, d_k//2)
        sin_table: (max_len, d_k//2)
    """
    half = d_k // 2
    i = torch.arange(0, half, dtype=torch.float32)           # (half,)
    freqs = 1.0 / (theta ** (2.0 * i / d_k))                # (half,) — θᵢ
    positions = torch.arange(max_len, dtype=torch.float32)   # (max_len,)
    angles = torch.outer(positions, freqs)                   # (max_len, half)
    return angles.cos(), angles.sin()


def apply_rope(
    x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor
) -> torch.Tensor:
    """
    对 Query 或 Key 张量应用 RoPE 旋转（LLaMA 半分割约定）。

    x  : (B, heads, T, d_k)
    cos: (T, d_k//2)  — 对应 T 个位置的 cos 值
    sin: (T, d_k//2)

    旋转：将 x 分成前后两半 x1, x2
      RoPE(x) = [x1*cos - x2*sin | x1*sin + x2*cos]

    注意：V 不旋转，位置信息只通过 Q·K 注意力权重传递。
    """
    d = x.shape[-1]
    x1, x2 = x[..., :d//2], x[..., d//2:]       # (B, heads, T, d//2)
    cos = cos.unsqueeze(0).unsqueeze(0)           # (1, 1, T, d//2)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return torch.cat([x1 * cos - x2 * sin,
                      x1 * sin + x2 * cos], dim=-1)


# ── 频率表可视化 ─────────────────────────────────────────────
D_K, MAX_LEN, THETA = 64, 64, 10000.0
cos_table, sin_table = precompute_rope_freqs(D_K, MAX_LEN, THETA)
angles = torch.arccos(cos_table.clamp(-1, 1))   # (max_len, d_k//2)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 角度热力图
ax = axes[0]
im = ax.imshow(angles.numpy(), aspect='auto', cmap='viridis', origin='lower')
plt.colorbar(im, ax=ax)
ax.set_xlabel('Dimension Pair (i)'); ax.set_ylabel('Position (m)')
ax.set_title(f'RoPE 角度热力图: m × θᵢ\n(d_k={D_K}, θ_base={THETA})', fontsize=11)

# 不同维度的角度随位置变化
ax2 = axes[1]
for dim_i, lw in zip([0, 2, 5, 10, 20, 31], [2.5, 2, 1.8, 1.5, 1.3, 1.2]):
    ax2.plot(angles[:, dim_i].numpy(), linewidth=lw, label=f'pair {dim_i}')
ax2.set_xlabel('Position (m)'); ax2.set_ylabel('Angle (rad)')
ax2.set_title('各维度对的旋转角度随位置变化\n(低维度对旋转慢，高维度对旋转快)', fontsize=11)
ax2.legend(fontsize=8, ncol=2); ax2.grid(True, alpha=0.3)

# 频率分布
ax3 = axes[2]
half = D_K // 2
i_vals = np.arange(half)
freqs_np = 1.0 / (THETA ** (2.0 * i_vals / D_K))
ax3.bar(i_vals, freqs_np, color='steelblue', alpha=0.8, edgecolor='k', linewidth=0.3)
ax3.set_xlabel('Dimension Pair Index (i)'); ax3.set_ylabel('Frequency θᵢ')
ax3.set_title(f'RoPE 频率分布\nθᵢ = 1 / (10000^(2i/{D_K}))', fontsize=11)
ax3.set_yscale('log'); ax3.grid(True, axis='y', alpha=0.3)
ax3.axhline(freqs_np[0], color='red', linestyle='--', alpha=0.5, label=f'max θ={freqs_np[0]:.3f}')
ax3.axhline(freqs_np[-1], color='blue', linestyle='--', alpha=0.5, label=f'min θ={freqs_np[-1]:.2e}')
ax3.legend(fontsize=8)

plt.suptitle('RoPE 频率分析', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('rope_frequencies.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'频率范围: {freqs_np[0]:.4f} ~ {freqs_np[-1]:.2e} rad/pos')
print(f'最低频 (pair 0): 走完一圈需要 {2*math.pi/freqs_np[0]:.0f} 个位置')
print(f'最高频 (pair {half-1}): 走完一圈需要 {2*math.pi/freqs_np[-1]:.0f} 个位置')

In [ ]:
# ── 2D 旋转直觉可视化 ─────────────────────────────────────────
# 在 d_k=2 的极简情形下，RoPE = 将向量绕原点旋转 m*θ 角

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左图：单位向量 [1,0] 在不同位置的旋转轨迹
ax = axes[0]
ax.add_patch(plt.Circle((0, 0), 1.0, fill=False, color='lightgray', linewidth=1.5))

theta_demo = 0.8   # 较大角度，使旋转效果明显
cos2d, sin2d = precompute_rope_freqs(d_k=2, max_len=10, theta=1.0/theta_demo)
q_init = torch.tensor([1.0, 0.0])
cmap_colors = plt.cm.plasma(np.linspace(0.1, 0.95, 10))

for m in range(10):
    c_m, s_m = cos2d[m, 0].item(), sin2d[m, 0].item()
    q_rot = torch.tensor([q_init[0]*c_m - q_init[1]*s_m,
                           q_init[0]*s_m + q_init[1]*c_m])
    color = cmap_colors[m]
    ax.annotate('', xy=(q_rot[0].item(), q_rot[1].item()), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    ax.text(q_rot[0].item()*1.18, q_rot[1].item()*1.18, f'm={m}',
            fontsize=9, ha='center', va='center', color=color, fontweight='bold')

ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.axhline(0, color='gray', alpha=0.3); ax.axvline(0, color='gray', alpha=0.3)
ax.set_xlabel('x₀', fontsize=11); ax.set_ylabel('x₁', fontsize=11)
ax.set_title('RoPE：同一向量在不同位置的旋转结果\n'
             f'(d_k=2, θ={theta_demo:.1f} rad/pos)',
             fontsize=11, fontweight='bold')
sm = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(0, 9))
plt.colorbar(sm, ax=ax, label='Position m')

# 右图：旋转如何产生位置感知的注意力
ax2 = axes[1]
theta_demo2 = 0.5
cos2d2, sin2d2 = precompute_rope_freqs(d_k=2, max_len=20, theta=1.0/theta_demo2)

# Q 固定在位置 10，K 从位置 0 到 19
q_fixed = torch.tensor([[[[1.0, 0.3]]]])   # (1, 1, 1, 2)
k_fixed = torch.tensor([[[[0.8, 0.6]]]])

q_pos = 10
scores_2d = []
for k_pos in range(20):
    q_r = apply_rope(q_fixed, cos2d2[q_pos:q_pos+1], sin2d2[q_pos:q_pos+1])
    k_r = apply_rope(k_fixed, cos2d2[k_pos:k_pos+1], sin2d2[k_pos:k_pos+1])
    scores_2d.append((q_r * k_r).sum().item())

colors2 = ['tomato' if p == q_pos else 'steelblue' for p in range(20)]
bars = ax2.bar(range(20), scores_2d, color=colors2, edgecolor='k', linewidth=0.5)
ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Key Position (n)', fontsize=11)
ax2.set_ylabel('Attention Score: RoPE(Q,10)·RoPE(K,n)', fontsize=10)
ax2.set_title(f'RoPE 注意力分数随 Key 位置的变化\n'
              f'(Q固定在位置{q_pos}，红色=自注意)', fontsize=11, fontweight='bold')
ax2.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('rope_rotation_intuition.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. 相对位置特性实验验证

理论保证：$\langle\text{RoPE}(\mathbf{q}, m), \text{RoPE}(\mathbf{k}, n)\rangle$ 只依赖相对位置 $(m - n)$，不依赖绝对位置 $m, n$。

下面用实验验证：固定相对距离 $r = m - n$，改变 $m$（不同绝对位置），结果应当相同。

In [ ]:
torch.manual_seed(0)

d_test = 64
cos_t, sin_t = precompute_rope_freqs(d_test, max_len=200, theta=10000.0)

# 固定 Q, K 向量（只测试位置效果）
q_vec = torch.randn(1, 1, 1, d_test)
k_vec = torch.randn(1, 1, 1, d_test)


def rope_score(q, k, pos_q, pos_k):
    q_r = apply_rope(q, cos_t[pos_q:pos_q+1], sin_t[pos_q:pos_q+1])
    k_r = apply_rope(k, cos_t[pos_k:pos_k+1], sin_t[pos_k:pos_k+1])
    return (q_r * k_r).sum().item() / math.sqrt(d_test)


# ── 表格验证 ─────────────────────────────────────────────────
test_offsets = [0, 1, 3, 7, 15, 31]
test_anchors = [20, 50, 80, 120, 160]

print('验证 RoPE 相对位置特性：相同 r=m-n，不同绝对位置 m，分数应相同')
print(f"\n{'r':>5} ", end='')
for m in test_anchors:
    print(f'  m={m:>3}', end='')
print(f'  {'最大差':>8}')
print('-' * (5 + len(test_anchors) * 8 + 10))

for r in test_offsets:
    scores = []
    for m in test_anchors:
        n = m - r
        if n < 0:
            scores.append(None)
            continue
        scores.append(rope_score(q_vec, k_vec, m, n))
    valid = [s for s in scores if s is not None]
    diff = max(valid) - min(valid) if len(valid) > 1 else 0
    print(f'{r:>5}', end='')
    for s in scores:
        print(f'  {s:>7.4f}' if s is not None else '      n/a', end='')
    print(f'  {diff:>8.2e}')

print('\n结论：同一 r 下各 m 的分数几乎完全相同（差值 ≈ 浮点误差）✓')

# ── 可视化 ───────────────────────────────────────────────────
max_r = 80
anchors_vis = [30, 60, 90, 120]
fig, ax = plt.subplots(figsize=(10, 4))

for m in anchors_vis:
    rs, ss = [], []
    for r in range(max_r):
        n = m - r
        if n < 0: break
        rs.append(r); ss.append(rope_score(q_vec, k_vec, m, n))
    ax.plot(rs, ss, linewidth=2, label=f'm={m}', alpha=0.8)

ax.set_xlabel('Relative Distance r = m - n', fontsize=12)
ax.set_ylabel('Attention Score', fontsize=12)
ax.set_title('RoPE 相对位置特性验证\n'
             '不同绝对位置 m 的曲线完全重合 → 分数只依赖 r', fontsize=12, fontweight='bold')
ax.legend(fontsize=10, title='Q 的绝对位置 m')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('rope_relative_property.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. RoPE 与 KV Cache 的天然兼容性

RoPE 是目前与 KV Cache **兼容性最好**的位置编码，原因如下：

### 为什么 RoPE 与 KV Cache 天然兼容？

```
Prefill 阶段（处理 Prompt [t₀, t₁, ..., tₚ]）：
  对位置 i 的 token：
    K_i = W_K(x_i)                  ← 内容投影
    K_i_rotated = RoPE(K_i, i)      ← 加入位置 i 的旋转
    Cache: K_cache[i] = K_i_rotated ← 存储已旋转的 K

Decode 阶段（生成位置 P 的 token）：
  K_P = W_K(x_P)                    ← 只计算新 token 的 K
  K_P_rotated = RoPE(K_P, P)        ← 加入位置 P 的旋转
  K_full = [K_cache; K_P_rotated]   ← 拼接（cache 已含正确位置）

  计算注意力：Q_P_rotated · K_full^T
  = RoPE(Q_P, P) · [RoPE(K_0, 0), ..., RoPE(K_{P-1}, P-1), RoPE(K_P, P)]^T
  → 每个分量 RoPE(Q_P,P)·RoPE(K_i,i) = f(q_P, k_i, P-i)  ← 正确相对距离 ✓
```

### 与绝对 PE + KV Cache 的对比

用绝对 PE 时，位置编码是加在**输入 embedding**上的，K, V 中已经隐含了绝对位置信息——这没问题。  
但绝对 PE 无法体现相对位置，而 RoPE 通过旋转实现了真正的相对位置感知，同时 KV Cache 的正确性自动保持。

In [ ]:
class CausalSelfAttentionRoPE(nn.Module):
    """
    因果自注意力 + RoPE + KV Cache

    与 CausalSelfAttentionKV 的关键区别：
    1. 不接受外部位置编码输入
    2. 在层内部通过旋转 Q, K 编码位置（V 不旋转）
    3. 已旋转的 K 可以直接缓存（位置信息 baked-in）
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        max_len: int = 4096,
        theta: float = 10000.0,
        dropout: float = 0.1,
    ):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.last_attn_weights = None

        # 预计算 RoPE 频率表（register_buffer：随模型保存，不参与梯度）
        cos_table, sin_table = precompute_rope_freqs(self.d_k, max_len, theta)
        self.register_buffer('rope_cos', cos_table)   # (max_len, d_k//2)
        self.register_buffer('rope_sin', sin_table)

    def forward(
        self,
        x: torch.Tensor,
        past_kv: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        """
        x      : (B, T, d_model)  T=1 in decode phase
        past_kv: (K_cache, V_cache)，K_cache 中已存储旋转后的 K
        """
        B, T, _ = x.shape

        Q = self.W_q(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)

        # 确定新 token 的绝对起始位置
        past_len = past_kv[0].size(2) if past_kv is not None else 0

        # 取出对应位置的 RoPE cos/sin
        cos = self.rope_cos[past_len : past_len + T]   # (T, d_k//2)
        sin = self.rope_sin[past_len : past_len + T]

        # ── 旋转 Q 和 K（编码位置信息）──────────────────────────
        Q = apply_rope(Q, cos, sin)   # Q: 用于查询，需要知道自己的位置
        K = apply_rope(K, cos, sin)   # K: 用于被查询，需要知道自己的位置
        # V 不旋转：V 只携带语义内容，位置通过 softmax(Q·K^T) 权重隐式传递

        # ── KV Cache 拼接（K 已含位置旋转，可直接 concat）──────
        if past_kv is not None:
            K_cache, V_cache = past_kv
            K = torch.cat([K_cache, K], dim=2)
            V = torch.cat([V_cache, V], dim=2)

        total_len = K.size(2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # 因果掩码（仅 Prefill 阶段 T>1 时需要）
        if T > 1:
            row = torch.arange(T, device=x.device).view(T, 1)
            col = torch.arange(total_len, device=x.device).view(1, total_len)
            causal_mask = col <= (past_len + row)
            scores = scores.masked_fill(
                ~causal_mask.unsqueeze(0).unsqueeze(0), float('-inf')
            )

        attn = F.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        self.last_attn_weights = attn.detach()

        out = torch.matmul(self.dropout(attn), V)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)

        new_kv = (K.detach(), V.detach()) if use_cache else None
        return self.W_o(out), new_kv


# 快速验证
rope_attn = CausalSelfAttentionRoPE(d_model=64, num_heads=4, max_len=128)

# Prefill
x_prefill = torch.randn(2, 6, 64)
out_pf, kv = rope_attn(x_prefill, use_cache=True)
print(f'Prefill 输出: {out_pf.shape}, K cache: {kv[0].shape}')

# Decode
x_decode = torch.randn(2, 1, 64)
out_dc, kv2 = rope_attn(x_decode, past_kv=kv, use_cache=True)
print(f'Decode 输出: {out_dc.shape}, K cache (updated): {kv2[0].shape}')

## 5. GPTDecoderRoPE 完整模型

与 KV Cache 教程中 `GPTDecoderKV` 的主要区别：

| 组件 | `GPTDecoderKV` | `GPTDecoderRoPE` |
|------|---------------|------------------|
| 位置编码 | `nn.Embedding(max_len, d_model)`（可学习绝对 PE） | ✗ 无（位置由 RoPE 提供） |
| 注意力层 | `CausalSelfAttentionKV` | `CausalSelfAttentionRoPE` |
| 位置感知 | 绝对位置 | 相对位置 |
| 超长外推 | 超出 `max_len` 崩溃 | 可外推（见第 7 节） |

In [ ]:
class GPTBlockRoPE(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, max_len=4096, theta=10000.0, dropout=0.1):
        super().__init__()
        self.attn = CausalSelfAttentionRoPE(d_model, num_heads, max_len, theta, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, past_kv=None, use_cache=False):
        attn_out, new_kv = self.attn(self.norm1(x), past_kv, use_cache)
        x = x + attn_out
        x = x + self.ff(self.norm2(x))
        return x, new_kv


class GPTDecoderRoPE(nn.Module):
    """
    GPT-style Decoder with RoPE + KV Cache

    核心改动：删除 pos_emb（无绝对位置嵌入），RoPE 在每个注意力层内提供位置信息。
    """

    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        num_heads: int,
        num_layers: int,
        d_ff: int,
        max_len: int = 4096,
        theta: float = 10000.0,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.max_len = max_len
        self.num_layers = num_layers
        self.token_emb = nn.Embedding(vocab_size, d_model)
        # ← 没有 pos_emb！位置信息完全由各层的 RoPE 提供
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            GPTBlockRoPE(d_model, num_heads, d_ff, max_len, theta, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(
        self,
        idx: torch.Tensor,
        past_kv_list: Optional[List] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[List]]:
        # 纯 token embedding，不加位置编码
        x = self.drop(self.token_emb(idx))

        new_kv_list = []
        for i, block in enumerate(self.blocks):
            past_kv = past_kv_list[i] if past_kv_list is not None else None
            x, new_kv = block(x, past_kv=past_kv, use_cache=use_cache)
            new_kv_list.append(new_kv)

        logits = self.lm_head(self.norm(x))
        return logits, (new_kv_list if use_cache else None)

    @torch.no_grad()
    def generate_no_cache(self, idx: torch.Tensor, max_new_tokens: int) -> torch.Tensor:
        """朴素自回归（每步完整前向传播，无 KV Cache）"""
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.max_len:])
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

    @torch.no_grad()
    def generate_with_cache(self, idx: torch.Tensor, max_new_tokens: int) -> torch.Tensor:
        """带 KV Cache 的自回归（每步只处理 1 个 token）"""
        logits, kv_cache = self(idx, use_cache=True)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        idx = torch.cat([idx, next_token], dim=1)
        for _ in range(max_new_tokens - 1):
            logits, kv_cache = self(next_token, past_kv_list=kv_cache, use_cache=True)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            idx = torch.cat([idx, next_token], dim=1)
        return idx


# ── 参数量对比 ──────────────────────────────────────────────
CFG = dict(vocab_size=500, d_model=128, num_heads=4, num_layers=4, d_ff=512, max_len=256)

model_rope = GPTDecoderRoPE(**CFG).to(DEVICE)

# 参照：等配置的绝对 PE 模型（pos_emb 有 256*128=32768 额外参数）
abs_pe_params = CFG['max_len'] * CFG['d_model']
rope_params = count_params(model_rope)

print(f'GPTDecoderRoPE 参数量: {rope_params:,}')
print(f'等效绝对 PE 版本估算: {rope_params + abs_pe_params:,}（多 {abs_pe_params:,} 个 pos_emb 参数）')
print(f'RoPE 无额外可学习参数 ✓（频率表是固定计算值）')

sample_ids = torch.randint(1, 500, (1, 8)).to(DEVICE)
logits_r, cache_r = model_rope(sample_ids, use_cache=True)
print(f'\n前向输出 shape: {logits_r.shape}')
print(f'Layer 0 K cache shape: {cache_r[0][0].shape}  (batch=1, heads=4, seq=8, d_k=32)')

## 6. 正确性验证

验证 `generate_no_cache` 与 `generate_with_cache` 对相同模型权重产生**完全相同**的输出。

RoPE 与 KV Cache 兼容的数学保证：
1. Prefill 时，位置 $i$ 的 K 被旋转了角度 $i \cdot \boldsymbol{\theta}$，存入 cache
2. Decode 时，新 token 的 K 被旋转角度 $(P+j) \cdot \boldsymbol{\theta}$，追加到 cache
3. 完整 K_full 中，每个位置的 K 都携带自己正确的旋转角度
4. 注意力分数 $\text{RoPE}(Q_m, m) \cdot \text{RoPE}(K_n, n) = f(q_m, k_n, m-n)$ ← 与全序列计算完全等价 ✓

In [ ]:
torch.manual_seed(7)

model_test = GPTDecoderRoPE(
    vocab_size=200, d_model=64, num_heads=4,
    num_layers=2, d_ff=256, max_len=128
).to(DEVICE).eval()

prompt = torch.randint(1, 200, (1, 10)).to(DEVICE)
N_NEW = 10

with torch.no_grad():
    out_no_cache   = model_test.generate_no_cache(prompt.clone(), N_NEW)
    out_with_cache = model_test.generate_with_cache(prompt.clone(), N_NEW)

print(f'Prompt       : {prompt[0].tolist()}')
print(f'No Cache     : {out_no_cache[0].tolist()}')
print(f'With KV Cache: {out_with_cache[0].tolist()}')

match = torch.all(out_no_cache == out_with_cache).item()
print(f'\n两版本输出完全一致: {match} ✓' if match else '\n❌ 输出不一致，请检查实现！')

# 额外验证：逐步检查中间 logits 是否一致
print('\n逐步验证中间 logits...')
prompt_full = prompt.clone()
for step in range(N_NEW):
    seq_so_far = out_no_cache[:, :len(prompt[0])+step+1]
    logits_ref, _ = model_test(seq_so_far)
    logit_ref = logits_ref[0, -1]
    # with cache: rebuild from scratch for simplicity
    # (full equivalence already proven by token match above)
    pred_ref = logit_ref.argmax().item()
    pred_cache = out_with_cache[0, len(prompt[0])+step].item()
    status = '✓' if pred_ref == pred_cache else '✗'
    print(f'  Step {step+1}: token={pred_cache} {status}')

## 7. 可视化：注意力分数衰减 & RoPE 频率热力图

In [ ]:
# ── 注意力分数随相对距离的衰减 ────────────────────────────────
# RoPE 天然产生"距离越远注意力越小"的隐式偏置

torch.manual_seed(42)
d_k_vis = 64
cos_vis, sin_vis = precompute_rope_freqs(d_k_vis, max_len=300, theta=10000.0)

N_TRIALS = 200
max_dist = 150
avg_scores = np.zeros(max_dist)
std_scores = np.zeros(max_dist)
all_scores = np.zeros((N_TRIALS, max_dist))

Q_POS = 150
for trial in range(N_TRIALS):
    q_r = torch.randn(1, 1, 1, d_k_vis)
    k_r = torch.randn(1, 1, 1, d_k_vis)
    q_rot = apply_rope(q_r, cos_vis[Q_POS:Q_POS+1], sin_vis[Q_POS:Q_POS+1])
    for dist in range(max_dist):
        k_pos = Q_POS - dist
        k_rot = apply_rope(k_r, cos_vis[k_pos:k_pos+1], sin_vis[k_pos:k_pos+1])
        s = (q_rot * k_rot).sum().item() / math.sqrt(d_k_vis)
        all_scores[trial, dist] = s

avg_scores = all_scores.mean(axis=0)
std_scores = all_scores.std(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(range(max_dist), avg_scores, 'b-', linewidth=2, label='Mean score')
ax.fill_between(range(max_dist),
                avg_scores - std_scores, avg_scores + std_scores,
                alpha=0.2, color='blue', label='±1 std')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Relative Distance (m - n)', fontsize=12)
ax.set_ylabel('Average Attention Score', fontsize=12)
ax.set_title(f'RoPE 隐式相对位置偏置\n(d_k={d_k_vis}, 平均 {N_TRIALS} 个随机 Q,K 对)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 不同 theta 的对比
ax2 = axes[1]
for theta_val, color, style in [
    (100, 'tomato', '-'),
    (1000, 'orange', '--'),
    (10000, 'steelblue', '-'),
    (100000, 'purple', ':'),
]:
    cos_th, sin_th = precompute_rope_freqs(d_k_vis, max_len=300, theta=float(theta_val))
    scores_th = np.zeros(max_dist)
    torch.manual_seed(0)
    q_r = torch.randn(1, 1, 1, d_k_vis)
    k_r = torch.randn(1, 1, 1, d_k_vis)
    q_rot = apply_rope(q_r, cos_th[Q_POS:Q_POS+1], sin_th[Q_POS:Q_POS+1])
    for dist in range(max_dist):
        k_pos = Q_POS - dist
        k_rot = apply_rope(k_r, cos_th[k_pos:k_pos+1], sin_th[k_pos:k_pos+1])
        scores_th[dist] = (q_rot * k_rot).sum().item() / math.sqrt(d_k_vis)
    ax2.plot(range(max_dist), scores_th, color=color, linestyle=style,
             linewidth=2, label=f'θ_base={theta_val}')

ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('Relative Distance (m - n)', fontsize=12)
ax2.set_ylabel('Attention Score', fontsize=12)
ax2.set_title('不同 θ_base 的位置衰减特性\n(θ_base 越小，近距离偏置越强)', fontsize=11, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle('RoPE 注意力分数随相对距离的衰减', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('rope_attn_decay.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── RoPE cos/sin 频率表热力图 ─────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

D_K_HM, MAX_HM = 64, 128
cos_hm, sin_hm = precompute_rope_freqs(D_K_HM, MAX_HM, theta=10000.0)

for ax, data, title, cmap in [
    (axes[0, 0], cos_hm.numpy(), 'cos(m·θᵢ)  [全局]', 'RdBu'),
    (axes[0, 1], sin_hm.numpy(), 'sin(m·θᵢ)  [全局]', 'RdBu'),
    (axes[1, 0], cos_hm[:32, :16].numpy(), 'cos(m·θᵢ)  [低频部分放大]', 'RdBu'),
    (axes[1, 1], cos_hm[:32, 16:].numpy(), 'cos(m·θᵢ)  [高频部分放大]', 'RdBu'),
]:
    im = ax.imshow(data, aspect='auto', cmap=cmap, vmin=-1, vmax=1, origin='lower')
    plt.colorbar(im, ax=ax, fraction=0.02)
    ax.set_xlabel('Dimension Pair (i)', fontsize=10)
    ax.set_ylabel('Position (m)', fontsize=10)
    ax.set_title(title, fontsize=11)

plt.suptitle(f'RoPE 频率表可视化 (d_k={D_K_HM}, max_len={MAX_HM}, θ_base=10000)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('rope_freq_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

print('观察：')
print('  低维度对（左侧）：频率低，旋转角度小，需要很多位置才旋转一圈 → 感知长距离关系')
print('  高维度对（右侧）：频率高，旋转角度大，很快就旋转一圈 → 感知短距离关系')

## 8. 长度外推能力对比

### 绝对 PE 的外推局限

- 可学习绝对 PE 是一张大小为 `(max_len, d_model)` 的查找表
- 只有训练时见过的位置（0 到 `max_len-1`）才有可靠的嵌入值
- 推理时若序列超过 `max_len`，要么报错（越界），要么使用未训练的 embedding（随机噪声）

### RoPE 的外推优势

- cos/sin 是平滑周期函数，超出训练长度的位置也能计算出合理的值
- 不同维度对的频率不同，高频维度已在短序列内多次循环，低频维度保持缓慢旋转
- **扩展方法**：YaRN、NTK-aware scaling 通过调整 $\theta_\text{base}$ 进一步提升超长外推能力

$$\theta_i^{\text{NTK}} = \left(\theta_{\text{base}} \cdot \frac{L'}{L}\right)^{2i/d_k}$$

In [ ]:
# ── 长度外推：RoPE vs 绝对 PE ──────────────────────────────────

TRAIN_LEN = 64
EXTRAP_LEN = 256
D_K_EX = 32

# RoPE 的 cos 频率（smooth continuation beyond training）
cos_train, _ = precompute_rope_freqs(D_K_EX, TRAIN_LEN, theta=10000.0)
cos_ext,   _ = precompute_rope_freqs(D_K_EX, EXTRAP_LEN, theta=10000.0)

# 模拟绝对 PE：训练时用随机值，训练后固定；超出范围只能用随机噪声
torch.manual_seed(99)
abs_pe_trained = torch.randn(TRAIN_LEN, D_K_EX)     # 训练时的 embedding
abs_pe_outofrange = torch.randn(EXTRAP_LEN - TRAIN_LEN, D_K_EX)  # 未见过的位置
abs_pe_full = torch.cat([abs_pe_trained, abs_pe_outofrange], dim=0)

fig, axes = plt.subplots(2, 2, figsize=(15, 8))

dim_pairs_to_show = [0, 2, 5, 10]
labels = [f'pair {i}' for i in dim_pairs_to_show]

# RoPE cos 全程（连续平滑）
ax = axes[0, 0]
for di, label in zip(dim_pairs_to_show, labels):
    ax.plot(range(EXTRAP_LEN), cos_ext[:, di].numpy(), linewidth=1.5, label=label)
ax.axvline(TRAIN_LEN, color='red', linestyle='--', linewidth=2, label=f'训练截止 ({TRAIN_LEN})')
ax.axvspan(TRAIN_LEN, EXTRAP_LEN, alpha=0.08, color='orange', label='外推区域')
ax.set_title('RoPE: 平滑延伸到训练长度以外\n(sin/cos 函数天然连续)', fontsize=11, fontweight='bold')
ax.set_xlabel('Position'); ax.set_ylabel('cos value')
ax.legend(fontsize=8, ncol=2); ax.grid(True, alpha=0.3)

# 绝对 PE（训练长度内 OK，之后是随机噪声）
ax2 = axes[0, 1]
for di, label in zip(dim_pairs_to_show, labels):
    ax2.plot(range(EXTRAP_LEN), abs_pe_full[:, di].numpy(), linewidth=1.5, label=label)
ax2.axvline(TRAIN_LEN, color='red', linestyle='--', linewidth=2, label=f'训练截止 ({TRAIN_LEN})')
ax2.axvspan(TRAIN_LEN, EXTRAP_LEN, alpha=0.15, color='red', label='Out-of-distribution\n（随机噪声）')
ax2.set_title('绝对 PE: 超出训练长度变为随机噪声\n(模型从未见过这些位置)', fontsize=11, fontweight='bold')
ax2.set_xlabel('Position'); ax2.set_ylabel('Embedding value')
ax2.legend(fontsize=8, ncol=2); ax2.grid(True, alpha=0.3)

# 平滑性对比（二阶差分）
ax3 = axes[1, 0]
diff_rope = np.diff(cos_ext[:, 0].numpy(), n=2)
diff_abs  = np.diff(abs_pe_full[:, 0].numpy(), n=2)
ax3.plot(range(len(diff_rope)), np.abs(diff_rope), 'b-', linewidth=1.5, label='RoPE (dim pair 0)', alpha=0.8)
ax3.plot(range(len(diff_abs)), np.abs(diff_abs),   'r-', linewidth=1.5, label='Abs PE (dim 0)',   alpha=0.8)
ax3.axvline(TRAIN_LEN, color='red', linestyle='--', linewidth=1.5)
ax3.set_yscale('log'); ax3.set_xlabel('Position')
ax3.set_ylabel('|2nd difference| (log scale)')
ax3.set_title('平滑性对比（二阶差分越小越平滑）', fontsize=11, fontweight='bold')
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)

# NTK-aware RoPE scaling（简介）
ax4 = axes[1, 1]
scale_factors = [1, 2, 4, 8]
for sf in scale_factors:
    theta_ntk = 10000.0 * sf
    cos_ntk, _ = precompute_rope_freqs(D_K_EX, EXTRAP_LEN, theta=theta_ntk)
    ax4.plot(range(EXTRAP_LEN), cos_ntk[:, 0].numpy(), linewidth=1.5,
             label=f'θ_base=10000×{sf} (NTK sf={sf})')
ax4.axvline(TRAIN_LEN, color='gray', linestyle='--', linewidth=1.5, label=f'Training end')
ax4.set_title('NTK-aware Scaling: 放大 θ_base 降低高频\n(提升超长序列外推能力)', fontsize=11, fontweight='bold')
ax4.set_xlabel('Position'); ax4.set_ylabel('cos value (dim pair 0)')
ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3)

plt.suptitle('长度外推能力对比：RoPE vs Absolute PE', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('rope_extrapolation.png', dpi=120, bbox_inches='tight')
plt.show()

# 定量：外推区域二阶差分均值
rope_smooth = np.abs(diff_rope[TRAIN_LEN:]).mean()
abs_smooth  = np.abs(diff_abs[TRAIN_LEN:]).mean()
print(f'外推区域平均二阶差分（越小越平滑）:')
print(f'  RoPE: {rope_smooth:.4f}')
print(f'  Abs PE: {abs_smooth:.4f}')
print(f'  平滑度改善: {abs_smooth/rope_smooth:.1f}x')

## 9. 总结

### RoPE 核心要点

| 特性 | 说明 |
|------|------|
| **位置编码方式** | 旋转 Q, K 向量（$R_m$ 旋转矩阵），V 不旋转 |
| **相对位置感知** | $\langle R_m q, R_n k\rangle = f(q, k, m-n)$，天然感知相对距离 |
| **KV Cache 兼容** | 旋转后的 K 可直接缓存，无需存储位置 index |
| **额外参数** | 无（频率表为固定计算值，不需要训练） |
| **长度外推** | sin/cos 连续，基础版可小幅外推；NTK/YaRN 可大幅扩展 |

### RoPE 实现要点回顾

```python
# 1. 预计算频率表
freqs_i = 1 / (10000 ** (2i / d_k))         # 几何递减频率
angle[m, i] = m * freqs_i                    # 位置 × 频率

# 2. 对 Q, K 应用旋转（V 不旋转！）
Q = apply_rope(W_q(x), cos[pos], sin[pos])   # 编码位置
K = apply_rope(W_k(x), cos[pos], sin[pos])   # 编码位置

# 3. KV Cache 安全直接存储旋转后的 K
K_full = cat([K_cache, K_new], dim=seq)       # 位置已 baked-in ✓
```

### 与 KV Cache 教程的知识衔接

```
标准 KV Cache（GPTDecoderKV）
  输入嵌入: token_emb(idx) + pos_emb(pos)   ← 绝对 PE 加在输入
  注意力:   Q·K^T（无位置感知）
  缓存:     未旋转的 K, V

RoPE + KV Cache（GPTDecoderRoPE）
  输入嵌入: token_emb(idx)                  ← 无绝对 PE
  注意力:   RoPE(Q,m) · RoPE(K,n)^T        ← 相对位置感知
  缓存:     已旋转的 K（含位置信息），V 不变
```

### 进阶方向

- **YaRN**：动态 NTK 缩放，LLaMA-3 长文本版使用
- **ALiBi**：在注意力分数上加线性惩罚，不同于 RoPE
- **LongRoPE**：非均匀位置插值
- **GQA + RoPE**：分组查询注意力 + 旋转编码（LLaMA-2/3 标准配置）

### 延伸阅读

- [RoFormer: Enhanced Transformer with Rotary Position Embedding](https://arxiv.org/abs/2104.09864)
- [YaRN: Efficient Context Window Extension of Large Language Models](https://arxiv.org/abs/2309.00071)
- [Extending Context Window of Large Language Models via Positional Interpolation](https://arxiv.org/abs/2306.15595)
- [LLaMA: Open and Efficient Foundation Language Models](https://arxiv.org/abs/2302.13971)